# Temporal PRD + Meeting Generation from Jira — Instance-Aware Generation Stage

This notebook implements the **generation stage only** for the agreed research pipeline.

It preserves the decisions made earlier:

- Each project lives in its own folder and has its own `all_300_tickets_full.json`.
- Jira is the temporal backbone.
- Weekly snapshots are created for every project, but **documents are not generated every week**.
- PRDs: **2–4 per project**, selected from meaningful temporal points.
- Meetings: **8–15 per project**, selected from event-rich weeks.
- Scheduling is deterministic and reproducible.
- Future Jira information is removed **before** any historical prompt is sent to the LLM.
- Jira descriptions are reconstructed as-of the generation cutoff using description changelog history where available.
- Current/future status, resolution, comments, changelog events, and other future information must not leak backward.
- PRD and meeting generation use the **strict prompts agreed earlier without changing their wording**.
- OpenAI is used only for generation; scheduling and temporal filtering are Python.
- Generated artifacts are saved separately from hidden generation metadata.
- `generation_manifest.json` is **never** supplied to the later extraction pipeline.
- A `snapshot_manifest.json` establishes the common weekly experimental clock for KG snapshots, XGBoost, labels, and GraphRAG.
- Generation is resumable: existing artifacts are skipped unless explicitly overwritten.
- Always run the complete 30-project pipeline in `DRY_RUN=True` first.

> **Optimized version:** the 30-project dry run is now profiling-only, and Jira timestamp parsing is cached/ISO-native. The strict PRD and meeting prompts are unchanged.


> **Why this version exists:** the completed 30-repository dry run showed repository spans of roughly 302–1041 weeks. Those are long-lived Jira repositories, not bounded enterprise project lifecycles. This notebook therefore inserts a deterministic **project-instance selection stage** before document scheduling. Generation and weekly KG snapshots operate on selected coherent activity windows rather than the full repository history.

The strict PRD and meeting system prompts remain unchanged.


In [15]:
import os
import shutil
from google.colab import drive

target_mount = '/content/drive'

# Unmount if there is an active/stale mount
!usr/bin/fusermount -u {target_mount} 2>/dev/null || true
!umount -f {target_mount} 2>/dev/null || true

# Safely delete any leftover files or folders inside the mountpoint directory
if os.path.exists(target_mount):
    try:
        shutil.rmtree(target_mount)
        print(f"Successfully cleared and removed {target_mount}")
    except Exception as e:
        print(f"Could not remove directory {target_mount}: {e}")

drive.mount(target_mount, force_remount=True)

Successfully cleared and removed /content/drive
Mounted at /content/drive


In [16]:
# CELL 1 — Install dependencies
!pip install -q --upgrade openai "pandas==2.2.3"

## 2. Mount Google Drive and configure paths

Change only `DATASET_ROOT` if your Drive path differs.

Expected input structure:

```
GC1_Working_Jira_Dataset_300/
├── 01_JBEAP/
│   └── all_300_tickets_full.json
├── 02_JBIDE/
│   └── all_300_tickets_full.json
...
```

In [17]:
# CELL 2 — Mount Drive and configure paths

from pathlib import Path

# CHANGE THIS ONLY IF YOUR ACTUAL DRIVE PATH DIFFERS.
DATASET_ROOT = Path(
    "/content/drive/MyDrive/GC1/GC1_Jira_Dataset"
)

# Generated data is deliberately separate from the source Jira folders.
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/GC1/GC1_Generated_PRD_Meetings"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())
print("Output root:", OUTPUT_ROOT)

Dataset root: /content/drive/MyDrive/GC1/GC1_Jira_Dataset
Exists: True
Output root: /content/drive/MyDrive/GC1/GC1_Generated_PRD_Meetings


## 3. OpenAI setup

Store your API key in **Colab → Secrets** with the name `ANTHROPIC_API_KEY`.

The model name is intentionally a configuration value rather than being hard-coded into the generation logic.

In [18]:
# CELL 3 — Claude via OpenAI SDK

from google.colab import userdata
from openai import OpenAI

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY was not found. Add it in Colab -> Secrets."
    )

client = OpenAI(
    api_key=ANTHROPIC_API_KEY,
    base_url="https://api.anthropic.com/v1/"
)

MODEL = "claude-sonnet-5-5"

print("Claude client configured via OpenAI SDK.")
print("Generation model:", MODEL)

Claude client configured via OpenAI SDK.
Generation model: claude-sonnet-5-5


## 4. Strict generation prompts

These are the agreed prompts. They are kept unchanged here.

In [19]:
# CELL 4 — STRICT PROMPTS (DO NOT MODIFY DURING THE EXPERIMENT)

PRD_SYSTEM_PROMPT = """
You are generating a realistic Product Requirements Document (PRD)
for an enterprise software project for use in a temporal project-risk
research dataset.

The PRD must represent what project participants plausibly knew,
planned, expected, or proposed at the document's timestamp.

============================================================
STRICT TEMPORAL RULE
============================================================

You may use ONLY information contained in the supplied context.

The supplied context has already been temporally filtered to contain
only information available on or before the PRD timestamp.

Do not infer or invent later Jira events, future failures, later
decisions, future delays, later blockers, or eventual outcomes.

Do not foreshadow events simply because they would be plausible.

============================================================
DOCUMENT REALISM
============================================================

Write this as a normal enterprise PRD.

Do NOT write it as:
- a Jira summary
- an incident report
- a knowledge graph
- an entity extraction dataset
- a retrospective
- a risk prediction report

Do not explicitly label text for machine extraction.

The document may naturally contain:

- project objectives
- background and context
- project scope
- functional requirements
- non-functional requirements
- features
- system components
- dependencies
- ownership/team information
- milestones
- target dates
- acceptance criteria
- assumptions
- constraints
- explicitly known risks
- unresolved questions

============================================================
REQUIREMENTS
============================================================

Use stable requirement identifiers where realistic:

REQ-001
REQ-002
REQ-003

Requirements must represent plausible planned product behavior or
constraints.

Do not turn every Jira bug into a requirement.

============================================================
RISKS
============================================================

Mention a risk only when it is supported by information available
at this timestamp.

Do NOT invent a risk merely because a later Jira issue exists.

Distinguish between:

- known fact
- assumption
- uncertainty
- planned work
- potential risk

============================================================
JIRA REFERENCES
============================================================

Do not artificially mention Jira identifiers.

A Jira issue may be referenced only if it already exists at the PRD
timestamp and referencing it would be natural in a real PRD.

============================================================
PRD REVISION
============================================================

If this is a later PRD version, preserve continuity with the supplied
previous PRD.

Do not rewrite the entire project history as though all information
was known from the beginning.

Reflect only changes justified by the supplied current context.

============================================================
OUTPUT
============================================================

Do not output JSON.

Do not explain your reasoning.

Output only the PRD document.
"""


MEETING_SYSTEM_PROMPT = """
You are generating a realistic enterprise software project meeting
transcript for use in a temporal project-risk research dataset.

The transcript represents what participants plausibly knew and
discussed at the meeting timestamp.

============================================================
STRICT TEMPORAL RULE
============================================================

Use ONLY information contained in the supplied context.

The supplied context has already been temporally filtered to contain
only information available on or before the meeting timestamp.

Do not introduce:

- future Jira changes
- future project outcomes
- future failures
- later blockers
- later decisions
- future delays

Do not foreshadow later events.

============================================================
REALISM
============================================================

Write a realistic conversation rather than a structured project
report.

Participants may:

- discuss requirements
- discuss existing Jira issues
- raise blockers
- raise potential risks
- question dependencies
- discuss technical components
- assign action items
- propose solutions
- make decisions
- disagree
- defer decisions
- revise previous decisions
- discuss milestone impact
- discuss scope changes
- express uncertainty

Not every statement should be factual or certain.

Real meetings contain:

- questions
- proposals
- uncertainty
- disagreement
- tentative commitments
- incomplete information

============================================================
DECISIONS
============================================================

Maintain a meaningful distinction between statements such as:

"We might delay the release."

and

"We have decided to delay the release."

Do not turn proposals into decisions.

============================================================
RISKS AND BLOCKERS
============================================================

Do not manufacture risks because later Jira events eventually show
that something failed.

A blocker must represent something currently preventing or materially
stopping progress.

A risk is an uncertain future condition that could negatively affect
the project.

============================================================
ACTION ITEMS
============================================================

Action items should occur naturally where appropriate.

They should identify:
- what needs to be done
- who is responsible, when known
- deadlines only where naturally justified

============================================================
OUTPUT FORMAT
============================================================

Meeting ID: <ID>
Meeting title: <title>
Date/time: <timestamp>
Participants: <comma-separated participants>

Transcript:

[HH:MM] Speaker: utterance

Do not output JSON.

Do not label statements as ENTITY, RISK, BLOCKER, DECISION,
RELATIONSHIP, etc.

Do not explain your reasoning.

Output only the meeting transcript.
"""

## 5. Reproducible generation configuration

The event-richness score below is used **only to decide when documents should be generated**.

In [20]:
# CELL 5 — Configuration

DEFAULT_CONFIG = {
    # --------------------------------------------------------
    # PROJECT-INSTANCE SELECTION
    # --------------------------------------------------------
    # Each input folder is a long-lived Jira repository. We select
    # one or (when independently rich enough) two non-overlapping
    # activity windows to act as bounded project instances.
    "instance_window_weeks": 36,
    "instance_step_weeks": 4,
    "max_instances_per_repository": 1,
    "min_instance_unique_tickets": 25,
    "secondary_instance_min_richness_ratio": 0.65,
    "instance_nonoverlap_gap_weeks": 8,

    "snapshot_frequency": "W-SUN",

    # Agreed document ranges
    "min_prds": 1,
    "max_prds": 1,
    "min_meetings": 2,
    "max_meetings": 2,

    # Avoid clustering generated artifacts too tightly
    "min_prd_gap_weeks": 6,
    "min_meeting_gap_weeks": 2,

    "prd_target_words": 1800,
    "meeting_target_words": 1400,

    # Limit prompt size while retaining rich context
    "max_context_tickets": 60,

    # Scheduling only — NOT ML features
    "weights": {
        "issue_created": 1.0,
        "status_change": 2.0,
        "priority_change": 2.0,
        "assignment_change": 1.5,
        "link_change": 2.0,
        "version_change": 1.5,
        "component_change": 1.0,
        "description_change": 0.5,
        "comment": 1.0,
        "resolved": 1.0,
        "due_date_change": 2.0,
    },
}

# IMPORTANT:
# First run all 30 projects with DRY_RUN=True.
DRY_RUN = True

# None = all discovered projects.
# For the first real API generation run, set PROJECT_LIMIT = 1.
PROJECT_LIMIT = None

# Existing generated documents are reused by default.
OVERWRITE = False

PROMPT_VERSION = "generation-v1.0"
GENERATION_CONTRACT_VERSION = "1.0"

In [21]:
# CELL 6 — Imports and helpers

import json
import re
import time
import hashlib
from collections import defaultdict
from pathlib import Path

import pandas as pd


from functools import lru_cache
from datetime import datetime, timezone

@lru_cache(maxsize=None)
def _parse_timestamp_cached(value):
    """Fast cached parser for Jira ISO-8601 timestamp strings."""
    if value is None or value == "":
        return None

    try:
        dt = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return pd.Timestamp(dt).tz_convert("UTC")
    except (ValueError, TypeError):
        try:
            return pd.to_datetime(value, utc=True)
        except Exception:
            return None


def parse_timestamp(value):
    """Accept strings and already-parsed timestamps without repeated inference."""
    if value is None or value == "":
        return None

    if isinstance(value, pd.Timestamp):
        if value.tzinfo is None:
            return value.tz_localize("UTC")
        return value.tz_convert("UTC")

    if isinstance(value, datetime):
        ts = pd.Timestamp(value)
        if ts.tzinfo is None:
            return ts.tz_localize("UTC")
        return ts.tz_convert("UTC")

    return _parse_timestamp_cached(str(value))


def occurred_by(value, cutoff):
    ts = parse_timestamp(value)
    return ts is not None and ts <= cutoff

def json_dump(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )


def text_dump(text, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def read_text(path):
    return Path(path).read_text(encoding="utf-8")


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

## 7. Discover the per-project Jira files

This notebook does **not** concatenate the projects. Each folder is treated as its own project input.

In [22]:
# CELL 7 — Project discovery

def discover_projects(dataset_root):
    dataset_root = Path(dataset_root)

    if not dataset_root.exists():
        raise FileNotFoundError(
            f"Dataset root does not exist: {dataset_root}"
        )

    projects = []

    for folder in sorted(dataset_root.iterdir()):
        if not folder.is_dir():
            continue

        ticket_file = folder / "all_300_tickets_full.json"

        if not ticket_file.exists():
            continue

        match = re.match(r"^(\\d+)_(.+)$", folder.name)

        if match:
            project_number = int(match.group(1))
            project_key_from_folder = match.group(2)
        else:
            project_number = None
            project_key_from_folder = folder.name

        projects.append({
            "project_number": project_number,
            "project_key_from_folder": project_key_from_folder,
            "folder": folder,
            "ticket_file": ticket_file,
        })

    return projects


def load_project_tickets(project):
    path = project["ticket_file"]

    with open(path, "r", encoding="utf-8") as f:
        tickets = json.load(f)

    if not isinstance(tickets, list):
        raise ValueError(
            f"{path} does not contain a JSON list."
        )

    return tickets


projects = discover_projects(DATASET_ROOT)

print(f"Discovered {len(projects)} project folders.")

discovery_rows = []

for p in projects:
    tickets = load_project_tickets(p)

    jira_keys = sorted({
        str(t.get("project_key"))
        for t in tickets
        if t.get("project_key") is not None
    })

    discovery_rows.append({
        "folder": p["folder"].name,
        "folder_project_key": p["project_key_from_folder"],
        "tickets": len(tickets),
        "jira_project_keys": ", ".join(jira_keys),
        "exactly_300": len(tickets) == 300,
    })

discovery_df = pd.DataFrame(discovery_rows)
display(discovery_df)

if len(projects) != 30:
    print(
        f"WARNING: expected 30 project folders, found {len(projects)}."
    )

Discovered 30 project folders.


,folder,folder_project_key,tickets,jira_project_keys,exactly_300
0,01_JBEAP,01_JBEAP,300,JBEAP,True
1,02_JBIDE,02_JBIDE,300,JBIDE,True
2,03_SAK,03_SAK,300,SAK,True
3,04_JBDS,04_JBDS,300,JBDS,True
4,05_MNG,05_MNG,300,MNG,True
5,06_SPARK,06_SPARK,300,SPARK,True
6,07_ENTESB,07_ENTESB,300,ENTESB,True
7,08_SPR,08_SPR,300,SPR,True
8,09_AMBARI,09_AMBARI,300,AMBARI,True
9,10_MC,10_MC,300,MC,True


## 8. Deterministic Jira event stream

This is used to build the weekly calendar and choose meaningful document-generation points.

In [23]:
# CELL 8 — Deterministic event construction

def add_event(events, timestamp, event_type, issue_key, payload=None):
    ts = parse_timestamp(timestamp)
    if ts is None:
        return

    events.append({
        "timestamp": ts,
        "event_type": event_type,
        "issue_key": issue_key,
        "payload": payload or {},
    })


def map_changelog_field(field):
    field = str(field or "").strip().lower()

    if field == "status":
        return "status_change"
    if field == "priority":
        return "priority_change"
    if field in {"assignee", "assignment"}:
        return "assignment_change"
    if field in {"link", "issue links"}:
        return "link_change"
    if "version" in field:
        return "version_change"
    if field == "component":
        return "component_change"
    if field == "description":
        return "description_change"
    if "due" in field:
        return "due_date_change"

    return "other_change"


def extract_jira_events(tickets):
    events = []

    for ticket in tickets:
        issue_key = ticket.get("issue_key")

        add_event(
            events,
            ticket.get("created_at"),
            "issue_created",
            issue_key,
            {
                "issue_type": ticket.get("issue_type"),
                "summary": ticket.get("summary"),
                "priority": ticket.get("priority"),
            },
        )

        if ticket.get("resolved_at"):
            add_event(
                events,
                ticket.get("resolved_at"),
                "resolved",
                issue_key,
                {"resolution": ticket.get("resolution")},
            )

        for status_change in ticket.get("status_history", []):
            add_event(
                events,
                status_change.get("changed_at"),
                "status_change",
                issue_key,
                {
                    "from": status_change.get("from_status"),
                    "to": status_change.get("to_status"),
                    "actor": status_change.get("changed_by"),
                },
            )

        for comment in ticket.get("comments", []):
            add_event(
                events,
                comment.get("created_at"),
                "comment",
                issue_key,
                {
                    "comment_id": comment.get("comment_id"),
                    "author": comment.get("author"),
                    "body": comment.get("body"),
                },
            )

        for change in ticket.get("full_changelog", []):
            timestamp = change.get("created")

            for item in change.get("items", []):
                event_type = map_changelog_field(item.get("field"))

                add_event(
                    events,
                    timestamp,
                    event_type,
                    issue_key,
                    {
                        "field": item.get("field"),
                        "from": item.get("fromString"),
                        "to": item.get("toString"),
                    },
                )

    events.sort(key=lambda x: x["timestamp"])
    return events


def determine_project_range(events):
    if not events:
        raise ValueError("No timestamped Jira events found.")

    return (
        min(e["timestamp"] for e in events),
        max(e["timestamp"] for e in events),
    )

## 9. Weekly snapshot calendar

A snapshot exists every week even when no PRD or meeting is generated that week.

In [24]:
# CELL 9 — Weekly buckets

def create_weekly_buckets(events, start, end, frequency="W-SUN"):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    # The first cutoff is the first configured week-ending boundary
    # on or after the project start.
    first_cutoff = pd.date_range(
        start=start.normalize(),
        periods=1,
        freq=frequency,
        tz="UTC",
    )[0]

    if first_cutoff < start:
        first_cutoff = first_cutoff + pd.Timedelta(days=7)

    boundaries = pd.date_range(
        start=first_cutoff,
        end=end.normalize() + pd.Timedelta(days=7),
        freq=frequency,
        tz="UTC",
    )

    if len(boundaries) == 0:
        boundaries = pd.DatetimeIndex([end])

    buckets = []
    previous = start

    for week_number, boundary in enumerate(boundaries, start=1):
        week_events = [
            event
            for event in events
            if previous <= event["timestamp"] <= boundary
        ]

        buckets.append({
            "week": week_number,
            "start": previous,
            "end": boundary,
            "events": week_events,
        })

        previous = boundary + pd.Timedelta(microseconds=1)

    return buckets


def score_week(bucket, weights):
    score = 0.0
    counts = defaultdict(int)

    for event in bucket["events"]:
        event_type = event["event_type"]
        counts[event_type] += 1
        score += weights.get(event_type, 0.25)

    return {
        "week": bucket["week"],
        "start": bucket["start"],
        "end": bucket["end"],
        "score": score,
        "event_count": len(bucket["events"]),
        "counts": dict(counts),
    }

## 10. Project-instance selection

The completed dry run showed that the 300 tickets in each folder span many years. We therefore **do not treat a Jira repository as one project**.

For each repository this stage scans fixed 36-week candidate windows (every 4 weeks), scores them using:
- number of unique tickets active in the window;
- Jira event richness;
- number of active weeks;
- continuity/activity density.

The richest eligible window is selected. A second non-overlapping instance is retained only when it is independently rich enough (at least 65% of the primary instance's richness). This gives approximately 30–60 possible instances, with the actual number determined by the data.

This score is **only for instance selection**. It is not an ML feature and is never fed to XGBoost.


In [25]:
# CELL 10 — Candidate project-instance construction and selection

def build_candidate_instances(events, config):
    if not events:
        return []

    window_weeks = config["instance_window_weeks"]
    step_weeks = config["instance_step_weeks"]

    first = min(e["timestamp"] for e in events).normalize()
    last = max(e["timestamp"] for e in events).normalize()

    candidates = []
    cursor = first
    candidate_number = 0

    while cursor <= last:
        window_start = cursor
        window_end = window_start + pd.Timedelta(weeks=window_weeks) - pd.Timedelta(microseconds=1)

        window_events = [
            e for e in events
            if window_start <= e["timestamp"] <= window_end
        ]

        unique_tickets = {
            e["issue_key"]
            for e in window_events
            if e.get("issue_key")
        }

        active_week_ids = {
            int((e["timestamp"] - window_start).days // 7)
            for e in window_events
        }

        event_count = len(window_events)
        ticket_count = len(unique_tickets)
        active_weeks = len(active_week_ids)
        activity_density = active_weeks / float(window_weeks)

        # Deterministic selection score only.
        richness = (
            2.0 * ticket_count
            + 0.35 * event_count
            + 3.0 * active_weeks
            + 20.0 * activity_density
        )

        candidate_number += 1

        candidates.append({
            "candidate_number": candidate_number,
            "start": window_start,
            "end": min(window_end, last),
            "unique_tickets": ticket_count,
            "event_count": event_count,
            "active_weeks": active_weeks,
            "activity_density": activity_density,
            "richness": richness,
            "ticket_keys": sorted(unique_tickets),
        })

        cursor = cursor + pd.Timedelta(weeks=step_weeks)

    return candidates


def windows_conflict(a, b, gap_weeks):
    gap = pd.Timedelta(weeks=gap_weeks)

    return not (
        a["end"] + gap < b["start"]
        or b["end"] + gap < a["start"]
    )


def select_project_instances(events, config):
    candidates = build_candidate_instances(events, config)

    eligible = [
        c for c in candidates
        if c["unique_tickets"] >= config["min_instance_unique_tickets"]
    ]

    # If the repository is unusually sparse, keep the richest candidate
    # rather than silently dropping the repository.
    pool = eligible if eligible else candidates

    if not pool:
        return [], candidates

    ranked = sorted(
        pool,
        key=lambda x: (
            x["richness"],
            x["unique_tickets"],
            x["event_count"],
        ),
        reverse=True,
    )

    primary = ranked[0]
    selected = [primary]

    if config["max_instances_per_repository"] > 1:
        threshold = (
            primary["richness"]
            * config["secondary_instance_min_richness_ratio"]
        )

        for candidate in ranked[1:]:
            if len(selected) >= config["max_instances_per_repository"]:
                break

            if candidate["richness"] < threshold:
                continue

            if any(
                windows_conflict(
                    candidate,
                    existing,
                    config["instance_nonoverlap_gap_weeks"],
                )
                for existing in selected
            ):
                continue

            selected.append(candidate)

    selected = sorted(selected, key=lambda x: x["start"])

    for i, instance in enumerate(selected, start=1):
        instance["instance_number"] = i

    return selected, candidates


def tickets_for_instance(tickets, instance):
    """
    Keep tickets that actually participate in the selected activity window.
    A ticket participates if it has at least one deterministic Jira event
    in the window. This avoids carrying unrelated repository history into
    the synthetic project instance.
    """
    start = instance["start"]
    end = instance["end"]
    selected_keys = set(instance["ticket_keys"])

    return [
        t for t in tickets
        if t.get("issue_key") in selected_keys
    ]


def events_for_instance(events, instance):
    return [
        e for e in events
        if instance["start"] <= e["timestamp"] <= instance["end"]
    ]


## 11. Deterministic PRD and meeting scheduling

- Initial PRD and kickoff are anchored near project start.
- Later artifacts are chosen from high-activity weeks.
- Minimum gaps prevent unrealistic clustering.
- The prompts do **not** decide how many documents to generate.

In [26]:
# CELL 11 — Scheduling

def select_spaced_weeks(
    candidates,
    count,
    min_gap_weeks,
    forced_weeks=None,
):
    forced_weeks = set(forced_weeks or [])
    selected = set(forced_weeks)

    ranked = sorted(
        candidates,
        key=lambda x: (x["score"], x["event_count"]),
        reverse=True,
    )

    for candidate in ranked:
        if len(selected) >= count:
            break

        week = candidate["week"]

        if any(
            abs(week - existing) < min_gap_weeks
            for existing in selected
        ):
            continue

        selected.add(week)

    # If spacing prevented reaching the target, fill from remaining
    # high-activity weeks while preserving unique weeks.
    if len(selected) < count:
        for candidate in ranked:
            if len(selected) >= count:
                break
            selected.add(candidate["week"])

    return sorted(selected)


def choose_prd_schedule(scored_weeks, config):
    total_weeks = len(scored_weeks)

    if total_weeks == 0:
        return []

    if total_weeks <= 12:
        desired = 2
    elif total_weeks <= 30:
        desired = 3
    else:
        desired = 4

    desired = max(
        config["min_prds"],
        min(desired, config["max_prds"], total_weeks),
    )

    selected = select_spaced_weeks(
        candidates=scored_weeks,
        count=desired,
        min_gap_weeks=config["min_prd_gap_weeks"],
        forced_weeks={1},
    )

    return [
        {
            "week": week,
            "version": version,
            "reason": (
                "initial_plan"
                if version == 1
                else "material_project_change"
            ),
        }
        for version, week in enumerate(selected, start=1)
    ]


def choose_meeting_schedule(scored_weeks, config):
    total_weeks = len(scored_weeks)

    if total_weeks == 0:
        return []

    desired = round(total_weeks / 3.5)

    desired = max(
        config["min_meetings"],
        min(desired, config["max_meetings"], total_weeks),
    )

    selected = select_spaced_weeks(
        candidates=scored_weeks,
        count=desired,
        min_gap_weeks=config["min_meeting_gap_weeks"],
        forced_weeks={1},
    )

    return [
        {
            "week": week,
            "meeting_number": i,
        }
        for i, week in enumerate(selected, start=1)
    ]


def infer_meeting_type(bucket, week):
    if week == 1:
        return "project kickoff and planning"

    counts = defaultdict(int)

    for event in bucket["events"]:
        counts[event["event_type"]] += 1

    if counts["link_change"] >= 2 or counts["status_change"] >= 5:
        return "dependency and delivery review"

    if counts["priority_change"] >= 2:
        return "risk and prioritisation review"

    if counts["version_change"] >= 2:
        return "release and milestone review"

    if counts["description_change"] >= 3:
        return "scope and requirements review"

    return "project planning and progress review"

## 12. Historical reconstruction — critical leakage protection

The LLM must not see the final/current ticket state when generating an earlier document.

This code:
- excludes tickets not yet created;
- removes future status changes;
- removes future comments;
- removes future changelog entries;
- hides future resolution;
- reconstructs descriptions as-of the cutoff where Jira changelog data permits it;
- derives status as-of cutoff.

The original Jira file is never modified.

In [27]:
# CELL 12 — Description reconstruction and historical ticket views

def description_as_of(ticket, cutoff):
    current_description = ticket.get("description")

    changes = []

    for change in ticket.get("full_changelog", []):
        timestamp = parse_timestamp(change.get("created"))

        if timestamp is None:
            continue

        for item in change.get("items", []):
            field = str(item.get("field", "")).strip().lower()

            if field != "description":
                continue

            changes.append({
                "timestamp": timestamp,
                "before": item.get("fromString"),
                "after": item.get("toString"),
            })

    changes.sort(key=lambda x: x["timestamp"])

    applicable = [
        change
        for change in changes
        if change["timestamp"] <= cutoff
    ]

    if applicable:
        return applicable[-1]["after"]

    future_changes = [
        change
        for change in changes
        if change["timestamp"] > cutoff
    ]

    if future_changes:
        # The first future edit's 'before' value is the best available
        # reconstruction of the description before that edit.
        return future_changes[0]["before"]

    return current_description


def value_as_of_from_changelog(
    ticket,
    cutoff,
    field_names,
    current_value,
):
    field_names = {
        str(x).strip().lower()
        for x in field_names
    }

    changes = []

    for change in ticket.get("full_changelog", []):
        timestamp = parse_timestamp(change.get("created"))

        if timestamp is None:
            continue

        for item in change.get("items", []):
            field = str(item.get("field", "")).strip().lower()

            if field not in field_names:
                continue

            changes.append({
                "timestamp": timestamp,
                "before": item.get("fromString"),
                "after": item.get("toString"),
            })

    changes.sort(key=lambda x: x["timestamp"])

    applicable = [
        c for c in changes
        if c["timestamp"] <= cutoff
    ]

    if applicable:
        return applicable[-1]["after"]

    future_changes = [
        c for c in changes
        if c["timestamp"] > cutoff
    ]

    if future_changes:
        return future_changes[0]["before"]

    return current_value


def filter_ticket_to_cutoff(ticket, cutoff):
    created = parse_timestamp(ticket.get("created_at"))

    if created is not None and created > cutoff:
        return None

    filtered_status_history = [
        x
        for x in ticket.get("status_history", [])
        if (
            occurred_by(x.get("changed_at"), cutoff)
        )
    ]

    filtered_comments = [
        x
        for x in ticket.get("comments", [])
        if (
            occurred_by(x.get("created_at"), cutoff)
        )
    ]

    filtered_changelog = [
        x
        for x in ticket.get("full_changelog", [])
        if (
            occurred_by(x.get("created"), cutoff)
        )
    ]

    status_as_of = value_as_of_from_changelog(
        ticket=ticket,
        cutoff=cutoff,
        field_names={"status"},
        current_value=ticket.get("status"),
    )

    priority_as_of = value_as_of_from_changelog(
        ticket=ticket,
        cutoff=cutoff,
        field_names={"priority"},
        current_value=ticket.get("priority"),
    )

    assignee_as_of = value_as_of_from_changelog(
        ticket=ticket,
        cutoff=cutoff,
        field_names={"assignee"},
        current_value=ticket.get("assignee"),
    )

    resolved_at = parse_timestamp(ticket.get("resolved_at"))

    if resolved_at is not None and resolved_at <= cutoff:
        historical_resolved_at = ticket.get("resolved_at")
        historical_resolution = ticket.get("resolution")
    else:
        historical_resolved_at = None
        historical_resolution = None

    return {
        "project_id": ticket.get("project_id"),
        "project_key": ticket.get("project_key"),
        "project_name": ticket.get("project_name"),

        "issue_id": ticket.get("issue_id"),
        "issue_key": ticket.get("issue_key"),
        "issue_type": ticket.get("issue_type"),

        "summary": ticket.get("summary"),
        "description": description_as_of(ticket, cutoff),

        "status_as_of_cutoff": status_as_of,
        "priority_as_of_cutoff": priority_as_of,
        "assignee_as_of_cutoff": assignee_as_of,

        "reporter": ticket.get("reporter"),
        "creator": ticket.get("creator"),
        "team": ticket.get("team"),

        "created_at": ticket.get("created_at"),
        "due_at": ticket.get("due_at"),
        "resolved_at": historical_resolved_at,
        "resolution": historical_resolution,

        "related_requirement": ticket.get("related_requirement"),
        "epic_feature": ticket.get("epic_feature"),

        "labels": ticket.get("labels", []),
        "components": ticket.get("components", []),

        # These arrays are filtered to the cutoff.
        "status_history": filtered_status_history,
        "comments": filtered_comments,
        "full_changelog": filtered_changelog,
    }


def build_context_at_cutoff(tickets, cutoff):
    historical = []

    for ticket in tickets:
        item = filter_ticket_to_cutoff(ticket, cutoff)

        if item is not None:
            historical.append(item)

    return historical

## 13. Context selection

Every prompt should not contain all 300 full tickets. This selector keeps the context temporally valid and favors active/information-rich tickets. The selection is only a prompt-size mechanism; it does not change the Jira source or the later KG.

In [28]:
# CELL 13 — Prompt context compression

def compress_context(tickets, cutoff, max_tickets=60):
    cutoff = pd.Timestamp(cutoff)

    def ticket_score(ticket):
        created = parse_timestamp(ticket.get("created_at"))

        if created is None:
            recency = 0.0
        else:
            age_days = max(
                0.0,
                (cutoff - created).total_seconds() / 86400.0,
            )
            recency = 1.0 / (1.0 + age_days / 30.0)

        activity = (
            len(ticket.get("comments", []))
            + len(ticket.get("status_history", []))
            + len(ticket.get("full_changelog", []))
            + len(ticket.get("components", []))
        )

        unresolved = 1.0 if ticket.get("resolved_at") is None else 0.0

        return (
            3.0 * recency
            + 0.25 * activity
            + 1.0 * unresolved
        )

    ranked = sorted(
        tickets,
        key=ticket_score,
        reverse=True,
    )

    selected = ranked[:max_tickets]

    compact = []

    for ticket in selected:
        compact.append({
            "issue_key": ticket.get("issue_key"),
            "issue_type": ticket.get("issue_type"),
            "summary": ticket.get("summary"),
            "description": ticket.get("description"),
            "status_as_of_cutoff": ticket.get("status_as_of_cutoff"),
            "priority_as_of_cutoff": ticket.get("priority_as_of_cutoff"),
            "assignee_as_of_cutoff": ticket.get("assignee_as_of_cutoff"),
            "created_at": ticket.get("created_at"),
            "due_at": ticket.get("due_at"),
            "resolved_at": ticket.get("resolved_at"),
            "components": ticket.get("components", []),
            "recent_comments": ticket.get("comments", [])[-3:],
            "recent_changes": ticket.get("full_changelog", [])[-5:],
        })

    return compact

## 14. Prompt builders

The strict system prompts above remain unchanged. These user prompts supply only the project-specific, temporally filtered context.

In [29]:
# CELL 14 — User prompt builders

def build_prd_user_prompt(
    project_key,
    project_name,
    cutoff,
    version,
    context,
    previous_prd,
    config,
):
    return f'''
PROJECT
=======

Project key: {project_key}
Project name: {project_name}

PRD version: {version}
PRD timestamp: {cutoff.isoformat()}

TARGET LENGTH
=============

Approximately {config["prd_target_words"]} words.

PREVIOUS PRD
============

{previous_prd if previous_prd else "None. This is the initial PRD."}

TEMPORALLY FILTERED PROJECT CONTEXT
===================================

The following Jira information contains ONLY information available
on or before {cutoff.isoformat()}.

{json.dumps(context, indent=2, ensure_ascii=False)}

Generate the PRD using the system instructions.
'''.strip()


def build_meeting_user_prompt(
    project_key,
    project_name,
    meeting_id,
    cutoff,
    meeting_type,
    context,
    previous_meetings,
    latest_prd,
    config,
):
    return f'''
PROJECT
=======

Project key: {project_key}
Project name: {project_name}

Meeting ID: {meeting_id}
Meeting timestamp: {cutoff.isoformat()}
Meeting type: {meeting_type}

TARGET LENGTH
=============

Approximately {config["meeting_target_words"]} words.

LATEST PRD AVAILABLE AT THIS TIME
=================================

{latest_prd if latest_prd else "No generated PRD available."}

PREVIOUS RELEVANT MEETINGS
==========================

{previous_meetings if previous_meetings else "None."}

TEMPORALLY FILTERED PROJECT CONTEXT
===================================

The following Jira information contains ONLY information available
on or before {cutoff.isoformat()}.

{json.dumps(context, indent=2, ensure_ascii=False)}

Generate the meeting transcript using the system instructions.
'''.strip()

## 15. OpenAI generation wrapper

Retries are included for transient API failures. No structured-output schema is used here because PRDs and meetings must remain realistic source artifacts rather than extraction answers.

In [30]:
# CELL 15 — OpenAI Responses API wrapper

def call_llm(
    system_prompt,
    user_prompt,
    model,
    max_retries=3,
):
    for attempt in range(max_retries):
        try:
          response = client.chat.completions.create(
          model=MODEL,
          messages=[
                  {
                      "role": "system",
                      "content": system_prompt
                  },
                  {
                      "role": "user",
                      "content": user_prompt
                  }
              ],
              max_tokens=8000
          )
          text = response.choices[0].message.content
          if not text or not text.strip():
                raise ValueError("Model returned empty text.")
          return text.strip()

        except Exception as e:
            if attempt == max_retries - 1:
                raise

            wait = 2 ** attempt

            print(
                f"OpenAI call failed (attempt {attempt + 1}/{max_retries}): {e}"
            )
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

## 16. Generate one project

The hidden manifest records which historical Jira context was available for each artifact. It is for provenance/evaluation only and must not enter the extraction pipeline.

In [31]:
# CELL 16 — Project generation

def generate_project(
    project,
    output_root,
    model,
    config,
    dry_run=True,
    overwrite=False,
):
    import time as _time

    t_project = _time.perf_counter()
    tickets = load_project_tickets(project)

    folder_key = project["project_key_from_folder"]

    jira_project_keys = sorted({
        str(t.get("project_key"))
        for t in tickets
        if t.get("project_key") is not None
    })

    project_key = (
        jira_project_keys[0]
        if len(jira_project_keys) == 1
        else folder_key
    )

    project_name = (
        tickets[0].get("project_name")
        if tickets
        else project_key
    ) or project_key

    print(f"Repository: {project_key}")
    print(f"Folder: {project['folder'].name}")
    print(f"Tickets: {len(tickets)}")

    t0 = _time.perf_counter()
    repository_events = extract_jira_events(tickets)
    repo_start, repo_end = determine_project_range(repository_events)
    print(
        f"  Repository span: {repo_start.date()} -> {repo_end.date()} "
        f"({((repo_end - repo_start).days / 7):.0f} weeks)"
    )
    print(f"  Event parsing: {_time.perf_counter() - t0:.2f}s")

    t0 = _time.perf_counter()
    selected_instances, all_candidates = select_project_instances(
        repository_events,
        config,
    )
    print(
        f"  Selected instances: {len(selected_instances)} "
        f"from {len(all_candidates)} candidate windows"
    )

    repository_dir = Path(output_root) / project["folder"].name
    repository_dir.mkdir(parents=True, exist_ok=True)

    repository_manifest = {
        "repository_key": project_key,
        "project_name": project_name,
        "source_folder": project["folder"].name,
        "source_ticket_file": str(project["ticket_file"]),
        "repository_ticket_count": len(tickets),
        "repository_start": repo_start.isoformat(),
        "repository_end": repo_end.isoformat(),
        "instance_selection_config": {
            k: config[k]
            for k in [
                "instance_window_weeks",
                "instance_step_weeks",
                "max_instances_per_repository",
                "min_instance_unique_tickets",
                "secondary_instance_min_richness_ratio",
                "instance_nonoverlap_gap_weeks",
            ]
        },
        "selected_instances": [],
        "warning": (
            "HIDDEN GENERATION METADATA. "
            "DO NOT PROVIDE THIS MANIFEST TO THE EXTRACTION PIPELINE."
        ),
    }

    instance_manifests = []

    for instance in selected_instances:
        instance_number = instance["instance_number"]
        instance_id = f"{project_key}-I{instance_number:02d}"

        instance_tickets = tickets_for_instance(tickets, instance)
        instance_events = events_for_instance(repository_events, instance)

        # Weekly snapshots cover the bounded instance, not the repository.
        buckets = create_weekly_buckets(
            instance_events,
            instance["start"],
            instance["end"],
            frequency=config["snapshot_frequency"],
        )

        scored = [
            score_week(bucket, config["weights"])
            for bucket in buckets
        ]

        prd_schedule = choose_prd_schedule(scored, config)
        meeting_schedule = choose_meeting_schedule(scored, config)

        print(
            f"    {instance_id}: "
            f"{instance['start'].date()} -> {instance['end'].date()} | "
            f"{len(instance_tickets)} tickets | "
            f"{len(instance_events)} events | "
            f"{len(buckets)} weekly snapshots | "
            f"PRDs scheduled: {len(prd_schedule)} | "
            f"Meetings scheduled: {len(meeting_schedule)}"
        )

        instance_dir = repository_dir / instance_id
        prd_dir = instance_dir / "prds"
        meeting_dir = instance_dir / "meetings"

        prd_dir.mkdir(parents=True, exist_ok=True)
        meeting_dir.mkdir(parents=True, exist_ok=True)

        snapshot_manifest = {
            "repository_key": project_key,
            "instance_id": instance_id,
            "project_name": project_name,
            "snapshot_frequency": config["snapshot_frequency"],
            "instance_start": instance["start"].isoformat(),
            "instance_end": instance["end"].isoformat(),
            "snapshots": [
                {
                    "week": bucket["week"],
                    "start": bucket["start"].isoformat(),
                    "cutoff": bucket["end"].isoformat(),
                    "jira_events_this_week": len(bucket["events"]),
                }
                for bucket in buckets
            ],
        }

        json_dump(
            snapshot_manifest,
            instance_dir / "snapshot_manifest.json",
        )

        manifest = {
            "repository_key": project_key,
            "instance_id": instance_id,
            "project_name": project_name,
            "source_folder": project["folder"].name,
            "source_ticket_file": str(project["ticket_file"]),
            "instance_ticket_count": len(instance_tickets),
            "instance_event_count": len(instance_events),
            "model": model,
            "prompt_version": PROMPT_VERSION,
            "generation_contract_version": GENERATION_CONTRACT_VERSION,
            "dry_run": dry_run,
            "selection": {
                "richness": instance["richness"],
                "unique_tickets": instance["unique_tickets"],
                "active_weeks": instance["active_weeks"],
                "activity_density": instance["activity_density"],
            },
            "timeline": {
                "start": instance["start"].isoformat(),
                "end": instance["end"].isoformat(),
                "snapshot_frequency": config["snapshot_frequency"],
                "weeks": len(buckets),
            },
            # Store the schedules even during dry-run so reporting is correct.
            "prd_schedule_planned": prd_schedule,
            "meeting_schedule_planned": meeting_schedule,
            "prd_schedule": [],
            "meeting_schedule": [],
            "weekly_profile": [
                {
                    "week": x["week"],
                    "start": x["start"].isoformat(),
                    "end": x["end"].isoformat(),
                    "score": x["score"],
                    "event_count": x["event_count"],
                    "counts": x["counts"],
                }
                for x in scored
            ],
            "warning": (
                "HIDDEN GENERATION METADATA. "
                "DO NOT PROVIDE THIS MANIFEST TO THE EXTRACTION PIPELINE."
            ),
        }

        # Genuine profiling-only dry run.
        if dry_run:
            json_dump(
                manifest,
                instance_dir / "generation_manifest.json",
            )
            instance_manifests.append(manifest)
            repository_manifest["selected_instances"].append({
                "instance_id": instance_id,
                "start": instance["start"].isoformat(),
                "end": instance["end"].isoformat(),
                "tickets": len(instance_tickets),
                "events": len(instance_events),
                "weeks": len(buckets),
                "prds_scheduled": len(prd_schedule),
                "meetings_scheduled": len(meeting_schedule),
                "richness": instance["richness"],
            })
            continue

        # ----------------------------------------------------
        # REAL GENERATION ONLY BELOW THIS POINT
        # ----------------------------------------------------

        generated_prds = {}
        previous_prd = None

        for scheduled in prd_schedule:
            week = scheduled["week"]
            version = scheduled["version"]
            bucket = buckets[week - 1]
            cutoff = bucket["end"]

            historical = build_context_at_cutoff(
                instance_tickets,
                cutoff,
            )

            context = compress_context(
                historical,
                cutoff=cutoff,
                max_tickets=config["max_context_tickets"],
            )

            document_id = f"{instance_id}-PRD-V{version}"

            prompt = build_prd_user_prompt(
                project_key=instance_id,
                project_name=project_name,
                cutoff=cutoff,
                version=version,
                context=context,
                previous_prd=previous_prd,
                config=config,
            )

            prompt_path = prd_dir / f"{document_id}.prompt.txt"
            output_path = prd_dir / f"{document_id}.txt"
            meta_path = prd_dir / f"{document_id}.meta.json"

            text_dump(PRD_SYSTEM_PROMPT + "\n\n" + prompt, prompt_path)

            if output_path.exists() and not overwrite:
                print(f"      Reusing existing {document_id}")
                prd_text = read_text(output_path)
            else:
                print(f"      Generating {document_id}...")
                prd_text = call_llm(
                    system_prompt=PRD_SYSTEM_PROMPT,
                    user_prompt=prompt,
                    model=model,
                )
                text_dump(prd_text, output_path)

            generated_prds[week] = prd_text
            previous_prd = prd_text

            context_issue_keys = [
                x["issue_key"]
                for x in context
                if x.get("issue_key")
            ]

            meta = {
                "repository_key": project_key,
                "instance_id": instance_id,
                "document_id": document_id,
                "document_type": "PRD",
                "model": model,
                "week": week,
                "knowledge_cutoff": cutoff.isoformat(),
                "prompt_version": PROMPT_VERSION,
                "generation_contract_version": GENERATION_CONTRACT_VERSION,
                "system_prompt_sha256": sha256_text(PRD_SYSTEM_PROMPT),
                "user_prompt_sha256": sha256_text(prompt),
                "context_issue_keys": context_issue_keys,
            }
            json_dump(meta, meta_path)

            manifest["prd_schedule"].append({
                "document_id": document_id,
                "week": week,
                "timestamp": cutoff.isoformat(),
                "reason": scheduled["reason"],
                "context_issue_keys": context_issue_keys,
                "output_file": str(output_path),
                "metadata_file": str(meta_path),
            })

        generated_meetings = []

        for scheduled in meeting_schedule:
            week = scheduled["week"]
            bucket = buckets[week - 1]
            cutoff = bucket["end"]

            historical = build_context_at_cutoff(
                instance_tickets,
                cutoff,
            )

            context = compress_context(
                historical,
                cutoff=cutoff,
                max_tickets=config["max_context_tickets"],
            )

            meeting_number = scheduled["meeting_number"]
            meeting_id = f"{instance_id}-M{meeting_number:03d}"
            meeting_type = infer_meeting_type(bucket, week)

            eligible_prds = [
                (prd_week, text)
                for prd_week, text in generated_prds.items()
                if prd_week <= week
            ]

            latest_prd = (
                max(eligible_prds, key=lambda x: x[0])[1]
                if eligible_prds
                else None
            )

            previous_meeting_text = "\n\n".join(
                generated_meetings[-2:]
            )

            prompt = build_meeting_user_prompt(
                project_key=instance_id,
                project_name=project_name,
                meeting_id=meeting_id,
                cutoff=cutoff,
                meeting_type=meeting_type,
                context=context,
                previous_meetings=previous_meeting_text,
                latest_prd=latest_prd,
                config=config,
            )

            prompt_path = meeting_dir / f"{meeting_id}.prompt.txt"
            output_path = meeting_dir / f"{meeting_id}.txt"
            meta_path = meeting_dir / f"{meeting_id}.meta.json"

            text_dump(
                MEETING_SYSTEM_PROMPT + "\n\n" + prompt,
                prompt_path,
            )

            if output_path.exists() and not overwrite:
                print(f"      Reusing existing {meeting_id}")
                meeting_text = read_text(output_path)
            else:
                print(f"      Generating {meeting_id}...")
                meeting_text = call_llm(
                    system_prompt=MEETING_SYSTEM_PROMPT,
                    user_prompt=prompt,
                    model=model,
                )
                text_dump(meeting_text, output_path)

            generated_meetings.append(meeting_text)

            context_issue_keys = [
                x["issue_key"]
                for x in context
                if x.get("issue_key")
            ]

            meta = {
                "repository_key": project_key,
                "instance_id": instance_id,
                "meeting_id": meeting_id,
                "document_type": "MEETING",
                "meeting_type": meeting_type,
                "model": model,
                "week": week,
                "knowledge_cutoff": cutoff.isoformat(),
                "prompt_version": PROMPT_VERSION,
                "generation_contract_version": GENERATION_CONTRACT_VERSION,
                "system_prompt_sha256": sha256_text(MEETING_SYSTEM_PROMPT),
                "user_prompt_sha256": sha256_text(prompt),
                "context_issue_keys": context_issue_keys,
            }
            json_dump(meta, meta_path)

            manifest["meeting_schedule"].append({
                "meeting_id": meeting_id,
                "week": week,
                "timestamp": cutoff.isoformat(),
                "meeting_type": meeting_type,
                "context_issue_keys": context_issue_keys,
                "output_file": str(output_path),
                "metadata_file": str(meta_path),
            })

        json_dump(
            manifest,
            instance_dir / "generation_manifest.json",
        )

        instance_manifests.append(manifest)

        repository_manifest["selected_instances"].append({
            "instance_id": instance_id,
            "start": instance["start"].isoformat(),
            "end": instance["end"].isoformat(),
            "tickets": len(instance_tickets),
            "events": len(instance_events),
            "weeks": len(buckets),
            "prds_scheduled": len(prd_schedule),
            "meetings_scheduled": len(meeting_schedule),
            "richness": instance["richness"],
        })

    json_dump(
        repository_manifest,
        repository_dir / "repository_instance_manifest.json",
    )

    print(f"  Repository total: {_time.perf_counter() - t_project:.2f}s")

    return {
        "repository": repository_manifest,
        "instances": instance_manifests,
    }


## 17. Run all discovered projects

A failure in one project is recorded and does not destroy the rest of the run.

In [32]:
# CELL 17 — Multi-project runner

def run_all_projects(
    dataset_root,
    output_root,
    model,
    config,
    dry_run=True,
    project_limit=None,
    overwrite=False,
):
    projects = discover_projects(dataset_root)

    print(f"Discovered {len(projects)} project(s).")

    if project_limit is not None:
        projects = projects[:project_limit]
        print(f"Limiting this run to {len(projects)} project(s).")

    manifests = {}

    for i, project in enumerate(projects, start=1):
        print()
        print("=" * 80)
        print(
            f"[{i}/{len(projects)}] "
            f"{project['folder'].name}"
        )
        print("=" * 80)

        try:
            manifest = generate_project(
                project=project,
                output_root=output_root,
                model=model,
                config=config,
                dry_run=dry_run,
                overwrite=overwrite,
            )

            manifests[project["folder"].name] = manifest

            repo_instances = manifest["repository"]["selected_instances"]
            print(
                f"  -> {len(repo_instances)} selected instance(s); "
                f"{sum(x['weeks'] for x in repo_instances)} weekly snapshots; "
                f"{sum(x['prds_scheduled'] for x in repo_instances)} PRDs scheduled; "
                f"{sum(x['meetings_scheduled'] for x in repo_instances)} meetings scheduled"
            )

        except Exception as e:
            print(f"FAILED: {project['folder'].name}")
            print(repr(e))

            manifests[project["folder"].name] = {
                "status": "FAILED",
                "error": repr(e),
            }

    json_dump(
        manifests,
        Path(output_root) / "all_generation_manifests.json",
    )

    return manifests

## 18. **RUN THIS FIRST:** full 30-project dry run

This performs no OpenAI generation. It creates schedules, snapshot manifests, prompts, and profiling information so the project timelines can be inspected before spending API tokens.

In [33]:
# CELL 18 — SAFE FIRST RUN

DRY_RUN = True
PROJECT_LIMIT = None
MODEL="model"
manifests = run_all_projects(
    dataset_root=DATASET_ROOT,
    output_root=OUTPUT_ROOT,
    model=MODEL,
    config=DEFAULT_CONFIG,
    dry_run=DRY_RUN,
    project_limit=PROJECT_LIMIT,
    overwrite=OVERWRITE,
)

Discovered 30 project(s).

[1/30] 01_JBEAP
Repository: JBEAP
Folder: 01_JBEAP
Tickets: 300
  Repository span: 2013-01-22 -> 2022-01-04 (467 weeks)
  Event parsing: 0.50s
  Selected instances: 1 from 117 candidate windows
    JBEAP-I01: 2015-11-24 -> 2016-08-01 | 131 tickets | 7087 events | 37 weekly snapshots | PRDs scheduled: 1 | Meetings scheduled: 2
  Repository total: 4.58s
  -> 1 selected instance(s); 37 weekly snapshots; 1 PRDs scheduled; 2 meetings scheduled

[2/30] 02_JBIDE
Repository: JBIDE
Folder: 02_JBIDE
Tickets: 300
  Repository span: 2004-10-09 -> 2022-01-03 (899 weeks)
  Event parsing: 0.14s
  Selected instances: 1 from 225 candidate windows
    JBIDE-I01: 2020-06-27 -> 2021-03-05 | 186 tickets | 2727 events | 37 weekly snapshots | PRDs scheduled: 1 | Meetings scheduled: 2
  Repository total: 5.53s
  -> 1 selected instance(s); 37 weekly snapshots; 1 PRDs scheduled; 2 meetings scheduled

[3/30] 03_SAK
Repository: SAK
Folder: 03_SAK
Tickets: 300
  Repository span: 2004-12-

## 19. Inspect the temporal profile before generation

Look especially for unusually long project timelines. A 300-ticket sample spanning many years may represent multiple development/release cycles rather than one coherent project instance and should be reviewed before synthetic generation.

In [34]:
# INSTANCE-AWARE profiling summary

def instance_summary(manifests):
    rows = []

    for folder_name, result in manifests.items():
        if result.get("status") == "FAILED":
            rows.append({
                "folder": folder_name,
                "repository": None,
                "instance_id": None,
                "start": None,
                "end": None,
                "tickets": None,
                "events": None,
                "weeks": None,
                "prds_scheduled": None,
                "meetings_scheduled": None,
                "richness": None,
                "status": "FAILED",
            })
            continue

        repo = result["repository"]

        for inst in repo["selected_instances"]:
            rows.append({
                "folder": folder_name,
                "repository": repo["repository_key"],
                "instance_id": inst["instance_id"],
                "start": inst["start"],
                "end": inst["end"],
                "tickets": inst["tickets"],
                "events": inst["events"],
                "weeks": inst["weeks"],
                "prds_scheduled": inst["prds_scheduled"],
                "meetings_scheduled": inst["meetings_scheduled"],
                "richness": round(inst["richness"], 2),
                "status": "OK",
            })

    return pd.DataFrame(rows)


summary = instance_summary(manifests)

display(
    summary.sort_values(
        ["folder", "instance_id"],
        na_position="last",
    )
)

ok = summary[summary["status"] == "OK"]

print()
print("Selected project instances:", len(ok))
print("Repositories represented:", ok["repository"].nunique())
print("Total weekly snapshots:", int(ok["weeks"].fillna(0).sum()))
print("Total scheduled PRDs:", int(ok["prds_scheduled"].fillna(0).sum()))
print("Total scheduled meetings:", int(ok["meetings_scheduled"].fillna(0).sum()))

print()
print("Instance weeks:")
display(ok["weeks"].describe().to_frame().T)

print("Instance ticket counts:")
display(ok["tickets"].describe().to_frame().T)


,folder,repository,instance_id,start,end,tickets,events,weeks,prds_scheduled,meetings_scheduled,richness,status
0,01_JBEAP,JBEAP,JBEAP-I01,2015-11-24T00:00:00+00:00,2016-08-01T23:59:59.999999+00:00,131,7087,37,1,2,2870.45,OK
1,02_JBIDE,JBIDE,JBIDE-I01,2020-06-27T00:00:00+00:00,2021-03-05T23:59:59.999999+00:00,186,2727,37,1,2,1404.67,OK
2,03_SAK,SAK,SAK-I01,2012-03-21T00:00:00+00:00,2012-11-27T23:59:59.999999+00:00,158,1071,37,1,2,818.85,OK
3,04_JBDS,JBDS,JBDS-I01,2016-03-04T00:00:00+00:00,2016-11-10T23:59:59.999999+00:00,229,3497,37,1,2,1809.95,OK
4,05_MNG,MNG,MNG-I01,2020-05-13T00:00:00+00:00,2021-01-19T23:59:59.999999+00:00,89,2066,37,1,2,1029.10,OK
5,06_SPARK,SPARK,SPARK-I01,2018-11-16T00:00:00+00:00,2019-07-25T23:59:59.999999+00:00,95,7293,37,1,2,2870.55,OK
6,07_ENTESB,ENTESB,ENTESB-I01,2020-04-08T00:00:00+00:00,2020-12-15T23:59:59.999999+00:00,257,5309,37,1,2,2500.15,OK
7,08_SPR,SPR,SPR-I01,2018-06-01T00:00:00+00:00,2019-01-14T00:00:00+00:00,300,675,34,1,2,1055.03,OK
8,09_AMBARI,AMBARI,AMBARI-I01,2016-03-03T00:00:00+00:00,2016-11-09T23:59:59.999999+00:00,105,3478,37,1,2,1555.30,OK
9,10_MC,MC,MC-I01,2020-04-01T00:00:00+00:00,2020-12-08T23:59:59.999999+00:00,183,12336,37,1,2,4811.60,OK



Selected project instances: 30
Repositories represented: 30
Total weekly snapshots: 1107
Total scheduled PRDs: 30
Total scheduled meetings: 60

Instance weeks:


,count,mean,std,min,25%,50%,75%,max
weeks,30.0,36.9,0.547723,34.0,37.0,37.0,37.0,37.0


Instance ticket counts:


,count,mean,std,min,25%,50%,75%,max
tickets,30.0,191.233333,73.794067,70.0,118.5,184.5,255.25,300.0


## 20. Inspect the exact schedule for one project

Do this before making real API calls.

In [35]:
# Inspect one selected project instance

SUCCESSFUL = [
    (folder, result)
    for folder, result in manifests.items()
    if result.get("status") != "FAILED"
    and result.get("instances")
]

if SUCCESSFUL:
    example_folder, example_result = SUCCESSFUL[0]
    example_manifest = example_result["instances"][0]

    print("Repository:", example_folder)
    print("Instance:", example_manifest["instance_id"])

    print("\nPlanned PRD schedule:")
    display(pd.DataFrame(example_manifest["prd_schedule_planned"]))

    print("\nPlanned meeting schedule:")
    display(pd.DataFrame(example_manifest["meeting_schedule_planned"]))

    print("\nWeekly profile:")
    display(pd.DataFrame(example_manifest["weekly_profile"]))


Repository: 01_JBEAP
Instance: JBEAP-I01

Planned PRD schedule:


,week,version,reason
0,1,1,initial_plan



Planned meeting schedule:


,week,meeting_number
0,1,1
1,8,2



Weekly profile:


,week,start,end,score,event_count,counts
0,1,2015-11-24T00:00:00+00:00,2015-11-29T00:00:00+00:00,54.75,42,"{'issue_created': 1, 'other_change': 13, 'comm..."
1,2,2015-11-29T00:00:00.000001+00:00,2015-12-06T00:00:00+00:00,383.75,282,"{'status_change': 140, 'version_change': 25, '..."
2,3,2015-12-06T00:00:00.000001+00:00,2015-12-13T00:00:00+00:00,421.00,320,"{'status_change': 118, 'other_change': 92, 're..."
3,4,2015-12-13T00:00:00.000001+00:00,2015-12-20T00:00:00+00:00,287.00,228,"{'status_change': 80, 'comment': 20, 'link_cha..."
4,5,2015-12-20T00:00:00.000001+00:00,2015-12-27T00:00:00+00:00,181.00,135,"{'comment': 19, 'status_change': 54, 'version_..."
5,6,2015-12-27T00:00:00.000001+00:00,2016-01-03T00:00:00+00:00,0.00,0,{}
6,7,2016-01-03T00:00:00.000001+00:00,2016-01-10T00:00:00+00:00,348.00,257,"{'comment': 36, 'status_change': 94, 'other_ch..."
7,8,2016-01-10T00:00:00.000001+00:00,2016-01-17T00:00:00+00:00,720.25,552,"{'status_change': 238, 'comment': 56, 'version..."
8,9,2016-01-17T00:00:00.000001+00:00,2016-01-24T00:00:00+00:00,532.50,482,"{'comment': 50, 'description_change': 3, 'stat..."
9,10,2016-01-24T00:00:00.000001+00:00,2016-01-31T00:00:00+00:00,320.75,274,"{'link_change': 20, 'comment': 54, 'other_chan..."


## Validate selected instances before any API generation

Do not enable the real-generation cell until the instance summary is sensible.

Expected behavior:
- approximately 30–60 selected instances overall;
- each instance is approximately 36 weeks by design;
- each instance has at least 25 unique active tickets unless a repository is too sparse and triggers the documented fallback;
- the second instance from a repository is selected only when it is non-overlapping and at least 65% as rich as the primary instance.

If these diagnostics are not reasonable, adjust the **instance-selection parameters**, rerun the dry run, and do not generate documents yet.


## 21. First real generation test — **one project only**

Run this only after the dry-run schedules look sensible.

This will make real OpenAI API calls.

In [36]:
# CELL 21 — ONE-PROJECT REAL GENERATION
# Uncomment only when ready.

DRY_RUN = False
PROJECT_LIMIT = 1

MODEL = "claude-sonnet-5-5"

test_generation = run_all_projects(
    dataset_root=DATASET_ROOT,
    output_root=OUTPUT_ROOT,
    model=MODEL,
    config=DEFAULT_CONFIG,
    dry_run=DRY_RUN,
    project_limit=PROJECT_LIMIT,
    overwrite=False,
)

Discovered 30 project(s).
Limiting this run to 1 project(s).

[1/1] 01_JBEAP
Repository: JBEAP
Folder: 01_JBEAP
Tickets: 300
  Repository span: 2013-01-22 -> 2022-01-04 (467 weeks)
  Event parsing: 0.65s
  Selected instances: 1 from 117 candidate windows
    JBEAP-I01: 2015-11-24 -> 2016-08-01 | 131 tickets | 7087 events | 37 weekly snapshots | PRDs scheduled: 1 | Meetings scheduled: 2
      Reusing existing JBEAP-I01-PRD-V1
      Reusing existing JBEAP-I01-M001
      Reusing existing JBEAP-I01-M002
  Repository total: 5.26s
  -> 1 selected instance(s); 37 weekly snapshots; 1 PRDs scheduled; 2 meetings scheduled


## 22. Full generation — all projects

Only run this after reviewing the first generated project's PRDs and meetings for temporal leakage, realism, continuity, and document quality.

Generation is resumable: existing `.txt` artifacts are reused when `overwrite=False`.

In [38]:
# CELL 22 — FULL REAL GENERATION
# Uncomment only after validating the one-project test.

DRY_RUN = False
PROJECT_LIMIT = None

final_generation = run_all_projects(
    dataset_root=DATASET_ROOT,
    output_root=OUTPUT_ROOT,
    model=MODEL,
    config=DEFAULT_CONFIG,
    dry_run=DRY_RUN,
    project_limit=PROJECT_LIMIT,
    overwrite=False,
)

Discovered 30 project(s).

[1/30] 01_JBEAP
Repository: JBEAP
Folder: 01_JBEAP
Tickets: 300
  Repository span: 2013-01-22 -> 2022-01-04 (467 weeks)
  Event parsing: 0.15s
  Selected instances: 1 from 117 candidate windows
    JBEAP-I01: 2015-11-24 -> 2016-08-01 | 131 tickets | 7087 events | 37 weekly snapshots | PRDs scheduled: 1 | Meetings scheduled: 2
      Reusing existing JBEAP-I01-PRD-V1
      Reusing existing JBEAP-I01-M001
      Reusing existing JBEAP-I01-M002
  Repository total: 7.02s
  -> 1 selected instance(s); 37 weekly snapshots; 1 PRDs scheduled; 2 meetings scheduled

[2/30] 02_JBIDE
Repository: JBIDE
Folder: 02_JBIDE
Tickets: 300
  Repository span: 2004-10-09 -> 2022-01-03 (899 weeks)
  Event parsing: 0.54s
  Selected instances: 1 from 225 candidate windows
    JBIDE-I01: 2020-06-27 -> 2021-03-05 | 186 tickets | 2727 events | 37 weekly snapshots | PRDs scheduled: 1 | Meetings scheduled: 2
      Reusing existing JBIDE-I01-PRD-V1
      Reusing existing JBIDE-I01-M001
      R

## 23. Expected output

Each project receives:

```
01_JBEAP/
├── repository_instance_manifest.json
├── JBEAP-I01/
│   ├── snapshot_manifest.json
│   ├── generation_manifest.json        # HIDDEN from extraction
│   ├── prds/
│   └── meetings/
└── JBEAP-I02/                          # only when independently rich enough
    ├── snapshot_manifest.json
    ├── generation_manifest.json
    ├── prds/
    └── meetings/
```

### What goes into the later extraction stage

Use:
- original Jira records;
- generated `*.txt` PRDs;
- generated `*.txt` meetings;
- `snapshot_manifest.json` as the common temporal clock.

Do **not** give the extraction model:
- `generation_manifest.json`;
- `*.meta.json`;
- stored generation prompts as source documents.

The later extraction stage maps Jira + PRDs + meetings into the agreed strict `ExtractionResult v1` contract, performs entity resolution, and materializes the Temporal Knowledge Graph. XGBoost features are then computed from weekly `KG(t)` snapshots, while GraphRAG uses the same graph and evidence to investigate SHAP-identified risk drivers.